# New York walkthrough — weekly Handle and GGR

Official source: [NY Gaming Commission revenue reports](https://gaming.ny.gov/revenue-reports).

New York publishes **weekly** mobile sports-wagering Excel files. We keep weeks as weeks.
We do **not** spread a week across calendar months.

| Term | Meaning |
| --- | --- |
| Handle | Amount wagered that week |
| GGR | Gross gaming revenue as reported by the Commission (stored in `gross_revenue`) |
| Hold | GGR / handle (analysis only) |

GGR is **cash basis** in New York. Futures can be taxed when written; winning tickets when redeemed.
That makes weekly GGR jump around. **Negative GGR is valid** and is preserved.

This notebook does **not** map operators to FanDuel / FLUT or produce a forecast.

## 1. Discover official workbook links

The landing page lists a STATEWIDE weekly Excel and one weekly Excel per operator.
Discovery uses the Sports Wagering table — no hard-coded dated `system/files` URLs.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from variant_gaming.common import http_get, project_root
from variant_gaming.states.new_york import LANDING_URL, discover_ny_sports_workbook_links, parse_ny_workbook
from variant_gaming.storage import connect, default_db_path

ROOT = project_root()
landing = http_get(LANDING_URL)
discovery = discover_ny_sports_workbook_links(landing.text)
print("Statewide:", discovery["statewide"]["discovered_url"])
print("Operators:", len(discovery["operators"]))
pd_ops = __import__("pandas").DataFrame(discovery["operators"])
pd_ops[["source_operator_name"]].head(12)

Statewide: https://gaming.ny.gov/statewide-sports-wagering-weekly-report-excel
Operators: 9


,source_operator_name
0,Bally Bet
1,BetMGM
2,Caesars Sport Book
3,DraftKings Sport Book
4,theScore Bet
5,Fanatics
6,FanDuel
7,Resorts World Bet
8,Rush Street Interactive


## 2. Parse a saved official sample

The production collector downloads each workbook, hashes it, and upserts weekly rows.
Here we parse the test fixture so the transformation stays visible without dumping thousands of rows.

In [2]:
fixture = ROOT / "tests" / "fixtures" / "NY" / "sample_statewide_weekly.xlsx"
parsed, sheet_check = parse_ny_workbook(fixture.read_bytes())
print("parsed rows", len(parsed))
print("sheet reconciliation rows", len(sheet_check))
print(parsed.columns.tolist())
parsed.head()

parsed rows 2
sheet reconciliation rows 1
['fiscal_year', 'week_ending', 'handle_usd', 'ggr_usd']


,fiscal_year,week_ending,handle_usd,ggr_usd
0,FY 2025/2026,2025-07-06,1000000.00,80000.00
1,FY 2025/2026,2025-07-13,2000000.50,-15000.25


## 3. Validation ideas (already implemented in the module)

- Completed weeks have handle and GGR
- Handle is positive; GGR may be negative
- Sheet weekly cells reconcile to the published Total
- Operator vs statewide comparison is labeled complete / incomplete / mismatch — never forced to balance
- Frequency stays `weekly`; `row_type` is `official_statewide_total` or `operator`

In [3]:
conn = connect(default_db_path(ROOT))
try:
    ny = __import__("pandas").read_sql_query(
        "SELECT row_type, COUNT(*) AS n, MIN(period_start) AS earliest, MAX(period_end) AS latest FROM gaming_results WHERE state_code='NY' GROUP BY row_type",
        conn,
    )
except Exception:
    ny = __import__("pandas").DataFrame()
conn.close()
ny

,row_type,n,earliest,latest
0,official_statewide_total,242,2022-01-03,2026-08-23
1,operator,2046,2022-01-03,2026-08-23


Full collection is run later in `20_run_all_collectors.ipynb`, which upserts and does not replace other states.